# GeneMiner on Google Colab (Free GPU)

This notebook starts your existing FastAPI backend in Colab and exposes it with ngrok.
It is intended for research/dev, not for always-on production serving.


In [ ]:
import os
import subprocess
import platform
from pathlib import Path

print("Python:", platform.python_version())
print("Platform:", platform.platform())

REPO_URL = 'https://github.com/YOUR_USERNAME/GeneMiner-DKD.git'
REPO_DIR = Path('/content/GeneMiner-DKD')
print("Repo URL:", REPO_URL)
print("Repo Dir:", REPO_DIR)


## 1) Clone or update repo


In [ ]:
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

if (REPO_DIR / '.git').exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull'], check=True)

os.chdir(REPO_DIR)


## 2) Runtime check
Enable GPU in Colab runtime for faster training, fallback to CPU is automatic.


In [ ]:
import torch

print('torch:', torch.__version__)
print('cuda_available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('torch_device:', torch.device('cuda'))
else:
    print('No CUDA. Will run on CPU.')
    print('torch_device:', torch.device('cpu'))


## 3) Install dependencies
Install package dependencies and ngrok helper.


In [ ]:
import sys

os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[api]'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyngrok'], check=True)
print('Dependencies installed')


## 4) Prepare import paths and validate app import


In [ ]:
from pathlib import Path
import sys

BACKEND_DIR = REPO_DIR / 'backend'
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

os.environ['PYTHONPATH'] = f"{BACKEND_DIR}:{os.environ.get('PYTHONPATH', '')}"
os.environ.setdefault('CORS_ORIGINS', '[\"http://localhost:5173\", \"http://127.0.0.1:5173\"]')

from app.main import app
print('App title:', app.title)


## 5) Start FastAPI in background


In [ ]:
import subprocess
import time
import requests

os.chdir(BACKEND_DIR)
server_proc = subprocess.Popen([
    'uvicorn', 'app.main:app', '--host', '0.0.0.0', '--port', '8000'
])
print('Backend PID:', server_proc.pid)
time.sleep(4)
try:
    r = requests.get('http://127.0.0.1:8000/health', timeout=20)
    print('Health:', r.status_code, r.text)
except Exception as e:
    print('Health check failed:', e)


## 6) Expose public URL by ngrok
Use an ngrok token for long sessions.


In [ ]:
from pyngrok import ngrok

# Optional: set token if required
# from getpass import getpass
# NGROK_TOKEN = getpass('Paste ngrok token: ')
# ngrok config add-authtoken <TOKEN>

public_tunnel = ngrok.connect(8000)
print('Public API URL:', public_tunnel.public_url)
print('Health check:', f'{public_tunnel.public_url}/health')


## 7) Use with React frontend
Set API base in your UI to the public URL shown above.


In [ ]:
BASE_URL = 'https://<your-ngrok-domain>'  # replace this
if '<your-ngrok-domain>' in BASE_URL:
    print('Replace BASE_URL with public_tunnel.public_url value.')
else:
    r = requests.get(f'{BASE_URL}/health', timeout=20)
    print(r.status_code, r.text)


## 8) Stop server and tunnel


In [ ]:
from pyngrok import ngrok

for t in ngrok.get_tunnels():
    ngrok.disconnect(t.public_url)

if 'server_proc' in globals() and server_proc.poll() is None:
    server_proc.terminate()
    print('Backend terminated.')

print('Cleanup done.')
